Next word prediction sequence

In [127]:
# Data Collection 
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/satyakibasu/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [128]:
data = gutenberg.raw('shakespeare-macbeth.txt')

# save to a text file
with open('macbeth.txt', 'w') as f:
    f.write(data)


Data Pre-processing

In [129]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer # to vectorize a text corpus
from tensorflow.keras.preprocessing.sequence import pad_sequences # all the sentences should be of same length
from sklearn.model_selection import train_test_split

In [130]:
# Load the dataset

with open('macbeth.txt', 'r') as f:
    text = f.read().lower()



# Tokenization - Creating word index
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1  # Adding 1 because of reserved 0 index
total_words

3553

In [131]:
# Covert every sentence to a sequence of tokens/indexes

input_sequences = []

# create input sequences using list of tokens
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

input_sequences

[[1, 885],
 [1, 885, 4],
 [1, 885, 4, 41],
 [1, 885, 4, 41, 57],
 [1, 885, 4, 41, 57, 1388],
 [1, 885, 4, 41, 57, 1388, 1389],
 [1, 885, 4, 41, 57, 1388, 1389, 1390],
 [418, 1391],
 [418, 1391, 1392],
 [418, 1391, 1392, 419],
 [270, 2],
 [270, 2, 886],
 [270, 2, 886, 33],
 [270, 2, 886, 33, 196],
 [270, 2, 886, 33, 196, 298],
 [81, 72],
 [81, 72, 38],
 [81, 72, 38, 32],
 [81, 72, 38, 32, 196],
 [81, 72, 38, 32, 196, 336],
 [81, 72, 38, 32, 196, 336, 131],
 [10, 270],
 [10, 270, 886],
 [10, 270, 886, 80],
 [10, 270, 886, 80, 10],
 [10, 270, 886, 80, 10, 1393],
 [128, 72],
 [128, 72, 1],
 [128, 72, 1, 1394],
 [128, 72, 1, 1394, 1395],
 [128, 72, 1, 1394, 1395, 84],
 [72, 1],
 [72, 1, 1396],
 [72, 1, 1396, 365],
 [72, 1, 1396, 365, 2],
 [72, 1, 1396, 365, 2, 887],
 [135, 7],
 [135, 7, 36],
 [135, 7, 36, 16],
 [135, 7, 36, 16, 172],
 [135, 7, 36, 16, 172, 1],
 [135, 7, 36, 16, 172, 1, 299],
 [135, 7, 36, 16, 172, 1, 299, 4],
 [135, 7, 36, 16, 172, 1, 299, 4, 666],
 [81, 76],
 [81, 76, 1],


In [132]:
# Apply pad sequences to make sure all sequences have same length
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
input_sequences

pd.DataFrame(input_sequences)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0,0,0,0,0,0,0,0,0,0,0,0,1,885
1,0,0,0,0,0,0,0,0,0,0,0,1,885,4
2,0,0,0,0,0,0,0,0,0,0,1,885,4,41
3,0,0,0,0,0,0,0,0,0,1,885,4,41,57
4,0,0,0,0,0,0,0,0,1,885,4,41,57,1388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15240,0,0,0,0,0,0,0,0,0,0,0,458,102,1173
15241,0,0,0,0,0,0,0,0,0,0,0,0,3552,1
15242,0,0,0,0,0,0,0,0,0,0,0,3552,1,885
15243,0,0,0,0,0,0,0,0,0,0,3552,1,885,4


In [133]:
## Create predictors and label
import tensorflow as tf

x,y = input_sequences[:,:-1],input_sequences[:,-1] # x is the input sequence and y is the label (last word). y is also the output feature

# pd.DataFrame(x) # for visulaization, convert to a dataframe and see what is being selected
#pd.DataFrame(y) # for visulaization, convert to a dataframe and see what is being selected

In [134]:
y = tf.keras.utils.to_categorical(y, num_classes=total_words)
#pd.DataFrame(y)

In [135]:
# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [136]:
# Define early stopping
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [137]:
# Train the LSTM RNN Model

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,GRU

## Define the model
model=Sequential()
model.add(Embedding(total_words,100,input_length=max_sequence_len-1))
model.add(LSTM(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words,activation="softmax"))

# #Compile the model
model.compile(loss="categorical_crossentropy",optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_14"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_14 (Embedding)    (None, 13, 100)           355300    
                                                                 
 lstm_25 (LSTM)              (None, 13, 150)           150600    
                                                                 
 dropout_13 (Dropout)        (None, 13, 150)           0         
                                                                 
 lstm_26 (LSTM)              (None, 100)               100400    
                                                                 
 dense_12 (Dense)            (None, 3553)              358853    
                                                                 
Total params: 965153 (3.68 MB)
Trainable params: 965153 (3.68 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [142]:
## Use the GRU Model
model=Sequential()
model.add(Embedding(total_words,100,input_length=max_sequence_len-1))
model.add(GRU(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(GRU(100))
model.add(Dense(total_words,activation="softmax"))

# #Compile the model
model.compile(loss="categorical_crossentropy",optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_15"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_15 (Embedding)    (None, 13, 100)           355300    
                                                                 
 gru (GRU)                   (None, 13, 150)           113400    
                                                                 
 dropout_14 (Dropout)        (None, 13, 150)           0         
                                                                 
 gru_1 (GRU)                 (None, 100)               75600     
                                                                 
 dense_13 (Dense)            (None, 3553)              358853    
                                                                 
Total params: 903153 (3.45 MB)
Trainable params: 903153 (3.45 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [143]:
## Train the model
history=model.fit(x_train,y_train,epochs=10,validation_data=(x_test,y_test),verbose=1,callbacks=[early_stopping])


Epoch 1/10
382/382 [==============================] - 6s 14ms/step - loss: 7.1742 - accuracy: 0.0339 - val_loss: 7.1130 - val_accuracy: 0.0246
Epoch 2/10
382/382 [==============================] - 5s 14ms/step - loss: 6.6910 - accuracy: 0.0360 - val_loss: 7.1311 - val_accuracy: 0.0348
Epoch 3/10
382/382 [==============================] - 6s 15ms/step - loss: 6.5104 - accuracy: 0.0392 - val_loss: 7.1081 - val_accuracy: 0.0472
Epoch 4/10
382/382 [==============================] - 5s 14ms/step - loss: 6.3478 - accuracy: 0.0465 - val_loss: 7.0863 - val_accuracy: 0.0453
Epoch 5/10
382/382 [==============================] - 5s 14ms/step - loss: 6.1915 - accuracy: 0.0482 - val_loss: 7.1358 - val_accuracy: 0.0482
Epoch 6/10
382/382 [==============================] - 5s 14ms/step - loss: 6.0231 - accuracy: 0.0540 - val_loss: 7.2607 - val_accuracy: 0.0508
Epoch 7/10
382/382 [==============================] - 5s 14ms/step - loss: 5.8310 - accuracy: 0.0580 - val_loss: 7.2725 - val_accuracy: 0.0515

In [139]:
# Predict the next word
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]

    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=-1)[0]
    
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None


In [140]:
input_text = "That will be ere the"
max_sequence_len = model.input_shape[1] + 1  # since input_length was max_sequence_len-1

next_word = predict_next_word(model, tokenizer, input_text, max_sequence_len)
print(f"Input Text: '{input_text}' --> Predicted Next Word: '{next_word}'")

Input Text: 'That will be ere the' --> Predicted Next Word: 'the'


In [141]:
# Save the model
model.save('lstm_text_generator_model.h5')

# Save the tokenizer since it is needed during inference as the tokenizer maps words to indexes used during training
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

/Users/satyakibasu/Documents/Satyaki/python_code/gen-ai/p311env/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
